In [15]:
import math

Rules of tumb:
- dmodel = 64 * nblocks
- dff = 4 * dmodel, if swiglu = 2.66 * dmodel
- chinchila optimal, if tokens/params > 20 also good, if less not gut

In [16]:
seq_len = 512
batch_size = 256
dmodel = 768
dff = 2.66 * dmodel
datt = dmodel
n_blocks = 12
q_heads = n_blocks
kv_heads = n_blocks
vocab_size = 128256

n_steps = 100

gpu_flops = 1.979 * 10**12  # H100 80GB
gpu_mem_bytes = 80 * 10**9  # H100 80GB
mfu = 0.1

bytes_per_param = 2  # fp16

In [17]:
print("--- Model Configuration ---")
print(f"seq_len: {seq_len}")
print(f"batch_size: {batch_size}")
print(f"dmodel: {dmodel}")
print(f"dff: {dff}")
print(f"datt: {datt}")
print(f"n_blocks: {n_blocks}")
print(f"q_heads: {q_heads}")
print(f"kv_heads: {kv_heads}")
print(f"vocab_size: {vocab_size}")

--- Model Configuration ---
seq_len: 512
batch_size: 256
dmodel: 768
dff: 2042.88
datt: 768
n_blocks: 12
q_heads: 12
kv_heads: 12
vocab_size: 128256


In [18]:
p_emb = vocab_size * dmodel
p_attn = 4 * dmodel**2
p_ff = 2 * dmodel * dff

total_params = n_blocks * (p_attn + p_ff) + 2 * p_emb
model_size_bytes = total_params * bytes_per_param

mem_required_bytes = 8 * total_params * bytes_per_param

tokens_in_dataset = batch_size * seq_len * n_steps
tokens_param_ratio = tokens_in_dataset / total_params
suggested_n_steps = math.ceil(20 * total_params / (batch_size * seq_len))

total_compute_needed = 6 * total_params * tokens_in_dataset

effective_flops = gpu_flops * mfu
seconds_to_train = total_compute_needed / effective_flops

print("--- Configuration ---")
print(f"Total Parameters: {total_params / 1e6:.2f} Million")
print(f"Tokens / Parameter Ratio (Chinchilla): {tokens_param_ratio:.2f}")
print(f"Suggested Training Steps (Chinchilla): {suggested_n_steps}")
print(f"Model Size (BF16): {model_size_bytes / 1e6:.2f} MB")
print(f"Estimated Training VRAM: {mem_required_bytes / 1e6:.2f} MB")
print(f"GPU Memory Utilization: {(mem_required_bytes / gpu_mem_bytes) * 100:.4f}%")
print(f"--- Training Time (for {tokens_in_dataset/1e6}M tokens) ---")
print(f"Time (using your FLOPS): {seconds_to_train / 3600:.2f} Hours")

--- Configuration ---
Total Parameters: 262.97 Million
Tokens / Parameter Ratio (Chinchilla): 0.05
Suggested Training Steps (Chinchilla): 40126
Model Size (BF16): 525.93 MB
Estimated Training VRAM: 4207.47 MB
GPU Memory Utilization: 5.2593%
--- Training Time (for 13.1072M tokens) ---
Time (using your FLOPS): 29.03 Hours
